# Agent Planning

**Module:** 10-agentic-ai-concepts

**Notebook:** `04-agent-planning.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **Planning Styles** with clear contracts and failure modes
- Explain and apply **Task Graphs** with clear contracts and failure modes
- Explain and apply **Plan-and-Execute Sketch** with clear contracts and failure modes
- Explain and apply **Replanning Triggers** with clear contracts and failure modes
- Explain and apply **Cost-Aware Planning** with clear contracts and failure modes
- Explain and apply **Verification Steps** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Agent Planning

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **Planning Styles**
2. **Task Graphs**
3. **Plan-and-Execute Sketch**
4. **Replanning Triggers**
5. **Cost-Aware Planning**
6. **Verification Steps**

Read top-to-bottom once, then revisit weak spots with the exercises.


## Planning Styles

### Definition
**Planning Styles** is a core building block in 04-agent-planning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Planning Styles typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Planning Styles: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Planning Styles as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Planning Styles as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Planning Styles
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Planning Styles when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does Planning Styles improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "Planning Styles" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Planning Styles"
    notebook: str = "04-agent-planning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


## Task Graphs

### Definition
**Task Graphs** is a core building block in 04-agent-planning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Task Graphs typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Task Graphs: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Task Graphs as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Task Graphs as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Task Graphs
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Task Graphs when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Task Graphs" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Task Graphs"
    notebook: str = "04-agent-planning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


### Worked scenario — Task Graphs

**Situation:** A team wants to productionize a feature involving **Task Graphs**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Plan-and-Execute Sketch

### Definition
**Plan-and-Execute Sketch** is a core building block in 04-agent-planning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Plan-and-Execute Sketch typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Plan-and-Execute Sketch: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Plan-and-Execute Sketch as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Plan-and-Execute Sketch as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Plan-and-Execute Sketch
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Plan-and-Execute Sketch when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Plan-and-Execute Sketch" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Plan-and-Execute Sketch"
    notebook: str = "04-agent-planning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


## Replanning Triggers

### Definition
**Replanning Triggers** is a core building block in 04-agent-planning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Replanning Triggers typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Replanning Triggers: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Replanning Triggers as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Replanning Triggers as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Replanning Triggers
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Replanning Triggers when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Replanning Triggers" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Replanning Triggers"
    notebook: str = "04-agent-planning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


### Worked scenario — Replanning Triggers

**Situation:** A team wants to productionize a feature involving **Replanning Triggers**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Cost-Aware Planning

### Definition
**Cost-Aware Planning** is a core building block in 04-agent-planning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Cost-Aware Planning typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Cost-Aware Planning: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Cost-Aware Planning as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Cost-Aware Planning as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Cost-Aware Planning
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Cost-Aware Planning when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Cost-Aware Planning" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Cost-Aware Planning"
    notebook: str = "04-agent-planning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
from enum import Enum

class Phase(Enum):
    PLAN = "plan"
    ACT = "act"
    CHECK = "check"
    DONE = "done"

def agent_loop(goal: str, max_steps=3):
    log = []
    phase = Phase.PLAN
    for step in range(max_steps):
        if phase == Phase.PLAN:
            plan = [f"step{i+1}" for i in range(2)]
            log.append(("plan", plan)); phase = Phase.ACT
        elif phase == Phase.ACT:
            log.append(("act", f"execute {goal}")); phase = Phase.CHECK
        elif phase == Phase.CHECK:
            log.append(("check", "ok")); phase = Phase.DONE
        else:
            break
    return log

print(agent_loop("triage ticket"))


In [ ]:
# Supervisor routes to specialists
def route_intent(text: str) -> str:
    t = text.lower()
    if "refund" in t or "invoice" in t: return "billing_agent"
    if "login" in t or "sso" in t: return "auth_agent"
    return "general_agent"

for q in ["refund please", "SSO loop", "hello"]:
    print(q, "->", route_intent(q))


## Verification Steps

### Definition
**Verification Steps** is a core building block in 04-agent-planning within agentic systems. Treat it as an operator with a limited toolbelt and a shift lead (you): something you can name, version, test, and operate.

### Why it matters
In agentic systems, weak designs around Verification Steps typically surface as runaway tool loops, unverifiable plans, and missing human gates. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Verification Steps: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like loops, memories, tools, and stop conditions.

### Intuition
Explain Verification Steps as an operator with a limited toolbelt and a shift lead (you). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Verification Steps as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Verification Steps
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of agentic systems: runaway tool loops, unverifiable plans, and missing human gates

### When to use
Use Verification Steps when your product path depends on this concern in agentic systems. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Verification Steps" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Verification Steps"
    notebook: str = "04-agent-planning"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Verification Steps"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Verification Steps"}
strong = {"definition": "Verification Steps", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Verification Steps"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Verification Steps", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Verification Steps

**Situation:** A team wants to productionize a feature involving **Verification Steps**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Comparison Snapshot

Use this table when reviewing designs in **Agent Planning**.

| Topic | Do | Don't |
|-------|----|-------|
| Planning Styles | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Task Graphs | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Plan-and-Execute Sketch | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Replanning Triggers | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Cost-Aware Planning | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Verification Steps | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| Planning Styles | Key concept covered in this notebook; see its section for definition and pitfalls |
| Task Graphs | Key concept covered in this notebook; see its section for definition and pitfalls |
| Plan-and-Execute Sketch | Key concept covered in this notebook; see its section for definition and pitfalls |
| Replanning Triggers | Key concept covered in this notebook; see its section for definition and pitfalls |
| Cost-Aware Planning | Key concept covered in this notebook; see its section for definition and pitfalls |
| Verification Steps | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Agent Planning** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **10-agentic-ai-concepts**.


## Try It Yourself

1. Implement a failing test/fixture for **Planning Styles**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Task Graphs**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Plan-and-Execute Sketch**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Replanning Triggers**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Cost-Aware Planning**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
